# RecSim overview

These examples use this checkout of New RecSim. From the repository root, create a virtual environment and run `python -m pip install -e ".[rl,docs]"`. Select that Python environment as your notebook kernel. All examples run on CPU and write only to temporary directories. The simulator itself does not require PyTorch.

Copyright 2019 The RecSim Authors. Licensed under the Apache License, Version 2.0.

## Components and interaction

A document sampler builds the candidates; a user model scores recommendations, simulates a response and updates hidden state. The Gymnasium adapter returns observations and rewards. Agents consume those observations and choose candidate **slot indices**. Document IDs are separately available in `info`.

In [ ]:
from recsim.environments import interest_evolution
from recsim.agents.random_agent import RandomAgent

config = dict(num_candidates=5, slate_size=2, seed=7, resample_documents=True, max_episode_steps=10)
env = interest_evolution.create_environment(config)
agent = RandomAgent(env.action_space, random_seed=17)
obs, info = env.reset(seed=7)
assert env.observation_space.contains(obs)
print("Candidate slots:", list(obs["doc"]))
print("Document IDs:", info["document_ids"])

## A reproducible episode

`reset` returns two values and `step` returns five. A natural session ending is `terminated`; an external step limit is `truncated`. Both stop the loop, but only a natural ending suppresses bootstrapping in a value-learning target.

In [ ]:
action = agent.begin_episode(obs)
rewards = []
while True:
    obs, reward, terminated, truncated, info = env.step(action)
    rewards.append(reward)
    assert env.observation_space.contains(obs)
    if terminated or truncated:
        agent.end_episode(reward, obs, terminated=terminated, truncated=truncated)
        break
    action = agent.step(reward, obs)
env.close()
print("Episode length:", len(rewards), "return:", sum(rewards))

## Training and evaluation

FullSlateQ treats each ordered slate as an action. Use small candidate sets because its action count grows combinatorially. The runner writes complete, versioned checkpoints at episode boundaries. These trusted local files contain policy, optimizer, replay and environment RNG state.

In [ ]:
from functools import partial
from pathlib import Path
from tempfile import TemporaryDirectory
from recsim.main import create_agent
from recsim.simulator.runner_lib import TrainRunner, EvalRunner

with TemporaryDirectory() as folder:
    factory = partial(create_agent, agent_name="full_slate_q", seed=7)
    common = dict(base_dir=folder, create_agent_fn=factory, max_steps_per_episode=10)
    train = TrainRunner(
        env=interest_evolution.create_environment(config),
        max_training_steps=10,
        num_iterations=1,
        **common,
    )
    assert train.run_experiment() == 10
    evaluate = EvalRunner(
        env=interest_evolution.create_environment(config), max_eval_episodes=2, **common
    )
    print("Evaluation returns:", evaluate.run_experiment())
    assert list((Path(folder) / "train" / "checkpoints").glob("*.pkl"))

## Simulation logs

CSV and JSONL preserve their historical schemas. The `user` field is the post-step observation, and `action` contains slot indices. `iter_episode` additionally provides both observations, acted-on document IDs and both termination flags. JSONL has one record per user, with that user’s steps in an array.

In [ ]:
import json
from recsim.simulation import simulate_users_jsonl

with TemporaryDirectory() as folder:
    path = simulate_users_jsonl(
        num_users=2,
        steps=3,
        slate_size=2,
        num_candidates=5,
        file_name=Path(folder) / "simulation.jsonl",
        progress=False,
    )
    records = [json.loads(line) for line in path.read_text().splitlines()]
    assert len(records) == 2
    print(records[0]["steps"][0]["action"])